# MiningSim dataset pattern ID standardisation

This notebook standardises the MiningSim sample datasets so the Blast to Mill workflow uses one consistent `pattern_id` convention across the bench.

The target convention is:

```text
A185-001-01, A185-001-02, A185-001-03, ...
```

It also renames files so the filename carries the same pattern identifier, for example:

```text
as_drilled/as_drilled_A185-001-01.csv
mwd/mwd_A185-001-01.csv
as_charged/as_charged_A185-001-01.csv
fms/fms_A185-001-01.csv
crusher/crusher_A185-001-01.csv
mill/sagmill_A185-001-01.csv
```

Place this notebook in the dataset root folder. The root folder should contain `blast_master.csv` and subfolders such as `as_drilled`, `mwd`, `as_charged`, `fms`, `crusher`, and `mill`.

> Note: `blast_master.csv` can be updated by this notebook, but for the attached sample set I have also supplied an already-corrected `blast_master.csv` directly.

In [1]:
from pathlib import Path
import pandas as pd
import re
import shutil
from datetime import datetime

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

## 1. Configuration

Update `BENCH_ID` only if the bench naming convention changes. The notebook is designed to be re-runnable: values that already look like `A185-001-01` will be normalised back to the same convention rather than being double-labelled.

In [2]:
# Root folder containing this notebook and blast_master.csv
ROOT = Path.cwd()

# New pattern naming convention
BENCH_ID = "A185-001"

# Set to True to preview the changes without writing any files
DRY_RUN = False

# Keep a copy of original files before overwriting/renaming
CREATE_BACKUPS = True

# Folder names expected under ROOT
FOLDERS = {
    "as_drilled": "as_drilled",
    "mwd": "mwd",
    "as_charged": "as_charged",
    "fms": "fms",
    "crusher": "crusher",
    "mill": "mill",
    "post_blast_fragmentation": "post_blast_fragmentation",
    "post_crusher_fragmentation": "post_crusher_fragmentation",
}

BACKUP_DIR = ROOT / f"_pattern_id_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

ROOT

WindowsPath('c:/GitHubRepositories/jupyter/dataset_generation/data')

## 2. Helper functions

The main helper accepts the known legacy formats used in the sample files:

- `1`, `2`, `3`
- `pattern_1`, `pattern_2`
- `A-185-08-01`, `A-185-08-02`
- already-correct values such as `A185-001-01`

In [3]:
def format_pattern_id(value, bench_id=BENCH_ID):
    """Convert a legacy pattern identifier to the standard BENCH_ID-## convention."""
    if pd.isna(value):
        return value

    s = str(value).strip()
    if not s:
        return s

    # Handles 1, 1.0, pattern_1, pattern-1, pattern 1.
    match = re.fullmatch(r"(?:pattern[_\s-]*)?(\d+)(?:\.0)?", s, flags=re.IGNORECASE)

    # Handles old pattern labels such as A-185-08-01 or already-correct A185-001-01.
    if not match:
        match = re.search(r"(\d{1,3})(?:\.0)?$", s)

    if not match:
        # Leave unexpected values unchanged so they can be reviewed rather than silently destroyed.
        return s

    number = int(match.group(1))
    return f"{bench_id}-{number:02d}"


def pattern_id_from_filename(path):
    """Infer a pattern ID from the trailing number in a filename."""
    stem = Path(path).stem
    match = re.search(r"(?:pattern[_\s-]*)?(\d+)$", stem, flags=re.IGNORECASE)
    if not match:
        match = re.search(r"(\d{1,3})$", stem)
    return format_pattern_id(match.group(1)) if match else None


def backup_file(path):
    """Copy the original file to the backup folder, preserving relative paths."""
    path = Path(path)
    if not CREATE_BACKUPS or DRY_RUN:
        return None
    relative = path.relative_to(ROOT)
    destination = BACKUP_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, destination)
    return destination


def discover_csv_files(folder_name, flat_globs=None):
    """Find CSV files in the intended folder layout, with a fallback for flat sample folders."""
    folder = ROOT / folder_name
    files = []

    if folder.exists():
        files.extend(sorted(folder.glob("*.csv")))

    # Fallback for flat sample extracts, if required.
    if flat_globs:
        for pattern in flat_globs:
            files.extend(sorted(ROOT.glob(pattern)))

    # De-duplicate while preserving order.
    seen = set()
    unique = []
    for file in files:
        resolved = file.resolve()
        if resolved not in seen and file.name != "blast_master.csv":
            seen.add(resolved)
            unique.append(file)
    return unique


def write_csv(df, path):
    if DRY_RUN:
        return
    df.to_csv(path, index=False)


def process_pattern_file(path, source_column, target_column, output_prefix):
    """Rename/normalise a pattern field and rename the file to prefix_PATTERNID.csv."""
    path = Path(path)
    df = pd.read_csv(path)

    if source_column not in df.columns and target_column not in df.columns:
        raise KeyError(f"{path.name}: expected '{source_column}' or '{target_column}', but found {list(df.columns)}")

    if source_column != target_column and source_column in df.columns:
        df = df.rename(columns={source_column: target_column})

    df[target_column] = df[target_column].map(format_pattern_id)
    pattern_values = sorted(v for v in df[target_column].dropna().unique())

    if len(pattern_values) == 0:
        pattern_id = pattern_id_from_filename(path)
    elif len(pattern_values) == 1:
        pattern_id = pattern_values[0]
    else:
        raise ValueError(f"{path.name}: contains multiple pattern_id values: {pattern_values}")

    if pattern_id is None:
        raise ValueError(f"{path.name}: could not infer pattern_id for output filename")

    new_path = path.with_name(f"{output_prefix}_{pattern_id}.csv")

    backup_file(path)

    if not DRY_RUN:
        write_csv(df, path)
        if new_path != path:
            if new_path.exists():
                new_path.unlink()
            path.rename(new_path)

    return {
        "source_file": str(path.relative_to(ROOT)),
        "new_file": str(new_path.relative_to(ROOT)),
        "rows": len(df),
        "pattern_id": pattern_id,
        "column": target_column,
    }


def rename_pattern_file_only(path, output_prefix):
    """Rename files that contain the pattern in the filename but do not have a pattern field."""
    path = Path(path)
    pattern_id = pattern_id_from_filename(path)
    if pattern_id is None:
        raise ValueError(f"{path.name}: could not infer pattern_id from filename")

    new_path = path.with_name(f"{output_prefix}_{pattern_id}.csv")
    backup_file(path)

    if not DRY_RUN and new_path != path:
        if new_path.exists():
            new_path.unlink()
        path.rename(new_path)

    return {
        "source_file": str(path.relative_to(ROOT)),
        "new_file": str(new_path.relative_to(ROOT)),
        "rows": None,
        "pattern_id": pattern_id,
        "column": "filename only",
    }


# Quick sanity check of the conversion logic.
test_values = [1, "2", "pattern_3", "A-185-08-04", "A185-001-05"]
{str(v): format_pattern_id(v) for v in test_values}

{'1': 'A185-001-01',
 '2': 'A185-001-02',
 'pattern_3': 'A185-001-03',
 'A-185-08-04': 'A185-001-04',
 'A185-001-05': 'A185-001-05'}

## 3. Update `blast_master.csv`

This updates the root-level `blast_master.csv` so patterns `1` through `27` become `A185-001-01` through `A185-001-27`.

In [4]:
blast_master_path = ROOT / "blast_master.csv"

if not blast_master_path.exists():
    raise FileNotFoundError(f"Could not find {blast_master_path}")

blast_master = pd.read_csv(blast_master_path)

if "pattern_id" not in blast_master.columns:
    raise KeyError("blast_master.csv must contain a pattern_id column")

original_pattern_ids = sorted(blast_master["pattern_id"].dropna().unique(), key=lambda x: str(x))
blast_master["pattern_id"] = blast_master["pattern_id"].map(format_pattern_id)
updated_pattern_ids = sorted(blast_master["pattern_id"].dropna().unique())

backup_file(blast_master_path)
write_csv(blast_master, blast_master_path)

print(f"Updated {blast_master_path.name}")
print(f"Original sample: {original_pattern_ids[:5]}")
print(f"Updated sample:  {updated_pattern_ids[:5]}")
print(f"Pattern count:   {len(updated_pattern_ids)}")

blast_master.head()

Updated blast_master.csv
Original sample: ['A185-001-01', 'A185-001-02', 'A185-001-03', 'A185-001-04', 'A185-001-05']
Updated sample:  ['A185-001-01', 'A185-001-02', 'A185-001-03', 'A185-001-04', 'A185-001-05']
Pattern count:   27


,pattern_id,pattern_type,point_id,x,y,floor_rl
0,A185-001-01,Ramp,1,363647.730715,6.591664e+06,185.0
1,A185-001-01,Ramp,2,363638.176051,6.591663e+06,185.0
2,A185-001-01,Ramp,3,363628.673178,6.591662e+06,185.0
3,A185-001-01,Ramp,4,363619.136434,6.591662e+06,185.0
4,A185-001-01,Ramp,5,363609.569439,6.591661e+06,185.0


## 4. Update each operational dataset

This section standardises field names and values, then renames files.

Expected changes:

| Dataset | Legacy field | Standard field | Output filename |
|---|---:|---:|---|
| As drilled | `pattern_id` | `pattern_id` | `as_drilled_A185-001-##.csv` |
| MWD | `polygon_id` | `pattern_id` | `mwd_A185-001-##.csv` |
| As charged | `pattern_number` | `pattern_id` | `as_charged_A185-001-##.csv` |
| FMS | `pattern_id` | `pattern_id` | `fms_A185-001-##.csv` |
| Crusher | `pattern` | `pattern_id` | `crusher_A185-001-##.csv` |
| Mill | `pattern` | `pattern_id` | `sagmill_A185-001-##.csv` |

In [5]:
processing_plan = [
    {
        "dataset": "as_drilled",
        "folder": FOLDERS["as_drilled"],
        "flat_globs": ["*as_drilled*.csv"],
        "source_column": "pattern_id",
        "target_column": "pattern_id",
        "output_prefix": "as_drilled",
    },
    {
        "dataset": "mwd",
        "folder": FOLDERS["mwd"],
        "flat_globs": ["mwd*.csv"],
        "source_column": "polygon_id",
        "target_column": "pattern_id",
        "output_prefix": "mwd",
    },
    {
        "dataset": "as_charged",
        "folder": FOLDERS["as_charged"],
        "flat_globs": ["as_charged*.csv"],
        "source_column": "pattern_number",
        "target_column": "pattern_id",
        "output_prefix": "as_charged",
    },
    {
        "dataset": "fms",
        "folder": FOLDERS["fms"],
        "flat_globs": ["fms*.csv"],
        "source_column": "pattern_id",
        "target_column": "pattern_id",
        "output_prefix": "fms",
    },
    {
        "dataset": "crusher",
        "folder": FOLDERS["crusher"],
        "flat_globs": ["crusher*.csv"],
        "source_column": "pattern",
        "target_column": "pattern_id",
        "output_prefix": "crusher",
    },
    {
        "dataset": "mill",
        "folder": FOLDERS["mill"],
        "flat_globs": ["sagmill*.csv", "mill*.csv"],
        "source_column": "pattern",
        "target_column": "pattern_id",
        "output_prefix": "sagmill",
    },
]

results = []

for item in processing_plan:
    files = discover_csv_files(item["folder"], item["flat_globs"])
    print(f"{item['dataset']}: found {len(files)} file(s)")

    for file in files:
        result = process_pattern_file(
            path=file,
            source_column=item["source_column"],
            target_column=item["target_column"],
            output_prefix=item["output_prefix"],
        )
        result["dataset"] = item["dataset"]
        results.append(result)

summary = pd.DataFrame(results)
summary

as_drilled: found 27 file(s)
mwd: found 27 file(s)
as_charged: found 27 file(s)
fms: found 27 file(s)
crusher: found 27 file(s)
mill: found 27 file(s)


,source_file,new_file,rows,pattern_id,column,dataset
0,as_drilled\blast_01_as_drilled.csv,as_drilled\as_drilled_A185-001-01.csv,51,A185-001-01,pattern_id,as_drilled
1,as_drilled\blast_02_as_drilled.csv,as_drilled\as_drilled_A185-001-02.csv,31,A185-001-02,pattern_id,as_drilled
2,as_drilled\blast_03_as_drilled.csv,as_drilled\as_drilled_A185-001-03.csv,88,A185-001-03,pattern_id,as_drilled
3,as_drilled\blast_04_as_drilled.csv,as_drilled\as_drilled_A185-001-04.csv,60,A185-001-04,pattern_id,as_drilled
4,as_drilled\blast_05_as_drilled.csv,as_drilled\as_drilled_A185-001-05.csv,71,A185-001-05,pattern_id,as_drilled
...,...,...,...,...,...,...
157,mill\sagmill_A-185-08-23.csv,mill\sagmill_A185-001-23.csv,253,A185-001-23,pattern_id,mill
158,mill\sagmill_A-185-08-24.csv,mill\sagmill_A185-001-24.csv,221,A185-001-24,pattern_id,mill
159,mill\sagmill_A-185-08-25.csv,mill\sagmill_A185-001-25.csv,100,A185-001-25,pattern_id,mill
160,mill\sagmill_A-185-08-26.csv,mill\sagmill_A185-001-26.csv,137,A185-001-26,pattern_id,mill


## 5. Optional: rename fragmentation files

The sample post-blast and post-crusher fragmentation files do not contain an explicit pattern field. However, their filenames still include the old pattern convention. This optional cell renames them to the same `A185-001-##` convention so folder contents remain consistent.

Leave this cell in place unless the fragmentation filenames are managed externally by another process.

In [6]:
fragmentation_results = []

fragmentation_plan = [
    {
        "dataset": "post_blast_fragmentation",
        "folder": FOLDERS["post_blast_fragmentation"],
        "flat_globs": ["post_blast_fragmentation*.csv"],
        "output_prefix": "post_blast_fragmentation",
    },
    {
        "dataset": "post_crusher_fragmentation",
        "folder": FOLDERS["post_crusher_fragmentation"],
        "flat_globs": ["post_crusher_fragmentation*.csv"],
        "output_prefix": "post_crusher_fragmentation_css130",
    },
]

for item in fragmentation_plan:
    files = discover_csv_files(item["folder"], item["flat_globs"])
    print(f"{item['dataset']}: found {len(files)} file(s)")

    for file in files:
        result = rename_pattern_file_only(file, item["output_prefix"])
        result["dataset"] = item["dataset"]
        fragmentation_results.append(result)

fragmentation_summary = pd.DataFrame(fragmentation_results)
fragmentation_summary

post_blast_fragmentation: found 27 file(s)
post_crusher_fragmentation: found 27 file(s)


,source_file,new_file,rows,pattern_id,column,dataset
0,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-01,filename only,post_blast_fragmentation
1,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-02,filename only,post_blast_fragmentation
2,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-03,filename only,post_blast_fragmentation
3,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-04,filename only,post_blast_fragmentation
4,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-05,filename only,post_blast_fragmentation
5,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-06,filename only,post_blast_fragmentation
6,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-07,filename only,post_blast_fragmentation
7,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-08,filename only,post_blast_fragmentation
8,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-09,filename only,post_blast_fragmentation
9,post_blast_fragmentation\post_blast_fragmentat...,post_blast_fragmentation\post_blast_fragmentat...,None,A185-001-10,filename only,post_blast_fragmentation


## 6. Validation checks

The checks below confirm that:

1. `blast_master.csv` contains the new `A185-001-##` labels.
2. Each processed dataset has a `pattern_id` column where expected.
3. No processed dataset still contains the legacy field names `polygon_id`, `pattern_number`, or `pattern`.

In [7]:
def validate_file(path, expected_column="pattern_id"):
    path = Path(path)
    df = pd.read_csv(path, nrows=1000)
    issues = []

    if expected_column not in df.columns:
        issues.append(f"missing {expected_column}")

    for legacy_col in ["polygon_id", "pattern_number", "pattern"]:
        if legacy_col in df.columns:
            issues.append(f"legacy column still present: {legacy_col}")

    if expected_column in df.columns:
        sample_values = [str(v) for v in df[expected_column].dropna().unique()[:10]]
        bad_values = [v for v in sample_values if not re.fullmatch(rf"{re.escape(BENCH_ID)}-\d{{2}}", v)]
        if bad_values:
            issues.append(f"unexpected pattern_id values: {bad_values}")

    return {
        "file": str(path.relative_to(ROOT)),
        "rows_checked": len(df),
        "issues": "; ".join(issues) if issues else "OK",
    }

validation_rows = []

# Re-discover after renaming.
for item in processing_plan:
    files = discover_csv_files(item["folder"], [f"{item['output_prefix']}_{BENCH_ID}-*.csv"])
    for file in files:
        validation_rows.append(validate_file(file))

validation = pd.DataFrame(validation_rows)
validation

,file,rows_checked,issues
0,as_drilled\as_drilled_A185-001-01.csv,51,OK
1,as_drilled\as_drilled_A185-001-02.csv,31,OK
2,as_drilled\as_drilled_A185-001-03.csv,88,OK
3,as_drilled\as_drilled_A185-001-04.csv,60,OK
4,as_drilled\as_drilled_A185-001-05.csv,71,OK
...,...,...,...
157,mill\sagmill_A185-001-23.csv,253,OK
158,mill\sagmill_A185-001-24.csv,221,OK
159,mill\sagmill_A185-001-25.csv,100,OK
160,mill\sagmill_A185-001-26.csv,137,OK


## 7. Final notes

After the notebook has run successfully:

- Use `pattern_id` consistently in downstream joins and filters.
- Avoid using filename parsing in the workflow where possible; use the `pattern_id` field inside each table as the source of truth.
- Keep `blast_master.csv` as the authoritative list of valid blast pattern polygons for the bench.
- Use the backup folder if you need to recover any original files.